# 01. Core Logging Fundamentals & Severity Levels

### Step 1: Why Logging Over `print()`?
In production Python applications, `print()` is non-persistent and cannot be managed by severity levels. The standard **`logging`** module offers structured tracking, runtime timestamps, severity filtering, and stack trace capture.

### Step 2: The 5 Standard Log Levels
1. **`DEBUG` (10)**: Granular diagnostic detail for developers during debugging.
2. **`INFO` (20)**: Routine application events confirming normal workflow.
3. **`WARNING` (30)**: Unexpected events or non-fatal performance drops.
4. **`ERROR` (40)**: Serious execution failure in a specific function/operation.
5. **`CRITICAL` (50)**: Severe failure indicating system crash or memory exhaustion.

In [8]:
import logging

# Reset root handlers to guarantee clear configuration on re-runs
logging.root.handlers = []

# 1. basicConfig setup for output layout
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s (%(filename)s:%(lineno)d): %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger("AppLogger")

# 2. Emitting logs across all 5 standard levels
logger.debug("DEBUG: Initializing memory buffers and database drivers.")
logger.info("INFO: User 'Vetri' successfully authenticated.")
logger.warning("WARNING: API response latency exceeded 500ms threshold.")
logger.error("ERROR: DB query failed due to unique key constraint violation.")
logger.critical(
    "CRITICAL: Out of Memory! Application state saving before shutdown."
)

2026-09-28 11:26:48 [DEBUG] AppLogger (3817515358.py:16): DEBUG: Initializing memory buffers and database drivers.
2026-09-28 11:26:48 [INFO] AppLogger (3817515358.py:17): INFO: User 'Vetri' successfully authenticated.
2026-09-28 11:26:48 [WARNING] AppLogger (3817515358.py:18): WARNING: API response latency exceeded 500ms threshold.
2026-09-28 11:26:48 [ERROR] AppLogger (3817515358.py:19): ERROR: DB query failed due to unique key constraint violation.
2026-09-28 11:26:48 [CRITICAL] AppLogger (3817515358.py:20): CRITICAL: Out of Memory! Application state saving before shutdown.


###  Code Explanation
* **`logging.root.handlers = []`**: Clears previous handlers attached to the root logger, preventing duplicated console lines when re-running Jupyter cells.
* **`format` string variables**:
  * `%(asctime)s`: Outputs the formatted timestamp based on `datefmt`.
  * `%(levelname)s`: Displays the level severity (`INFO`, `ERROR`, etc.).
  * `%(filename)s:%(lineno)d`: Identifies the exact file and line number that triggered the log.
* **`logger = logging.getLogger("AppLogger")`**: Instantiates a named logger scope instead of polluting the global root logger.

# 02. Advanced Logging: Handlers, Formatters & Rotating Files

### Step 1: Architecture Overview
* **Handlers**: Route logs to different targets simultaneously (`StreamHandler` -> Console, `RotatingFileHandler` -> Disk).
* **Formatters**: Define structural string patterns for each handler independently.
* **Log Rotation**: `RotatingFileHandler` automatically archives and creates new log files once size limits (`maxBytes`) are crossed to prevent disk space saturation.

In [10]:
import logging
from logging.handlers import RotatingFileHandler

app_logger = logging.getLogger("EnterpriseLogger")
app_logger.setLevel(logging.DEBUG)
app_logger.propagate = False

# Clear stale handlers on cell re-runs
if app_logger.hasHandlers():
    app_logger.handlers.clear()

# 1. Console Handler (Shows INFO and higher)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(
    logging.Formatter("[CONSOLE] %(levelname)s - %(message)s")
)

# 2. Rotating File Handler (Captures DEBUG and higher up to 1 KB per file)
log_filename = "app_rotating.log"
file_handler = RotatingFileHandler(
    log_filename, maxBytes=1024, backupCount=3, encoding="utf-8"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s [%(levelname)s] %(filename)s:%(lineno)d - %(message)s"
    )
)

# 3. Attach both handlers
app_logger.addHandler(console_handler)
app_logger.addHandler(file_handler)

# Simulate system activity to trigger file rotation
for i in range(12):
    app_logger.debug(
        f"Processing item index #{i+1} in background thread worker."
    )
    if i % 4 == 0:
        app_logger.info(f"Batch checkpoint #{i+1} completed successfully.")

print(
    f"Logs successfully written. Check directory for '{log_filename}' and backups."
)

[CONSOLE] INFO - Batch checkpoint #1 completed successfully.
[CONSOLE] INFO - Batch checkpoint #5 completed successfully.
[CONSOLE] INFO - Batch checkpoint #9 completed successfully.


Logs successfully written. Check directory for 'app_rotating.log' and backups.


###  Code Explanation 
* **`app_logger.propagate = False`**: Disables passing log messages up to the root logger, eliminating double-printing issues in console.
* **`maxBytes=1024` & `backupCount=3`**: When `app_rotating.log` reaches 1 KB (1024 bytes), Python renames it to `app_rotating.log.1` and creates a fresh `app_rotating.log`. It maintains up to 3 historic backups before overwriting the oldest.